# Assigment 3

In [1]:
import igl
import numpy as np
import pyvista as pv
from collections import deque #added by meeeee for task1.4
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

def plot_mesh_with_normals(V, F, N, factor=0.1):
    arrows = pv.vector_poly_data(V, N)
    arrows = arrows.glyph(orient='vectors',scale='mag',factor=factor)

    p = pv.Plotter()
    p.add_mesh(to_pyvista_mesh(V, F), show_edges=True)
    p.add_mesh(arrows)
    p.show()

def plot_lines(p, v0, v1, color):
    tmp = np.zeros((v0.shape[0] * 2, 3))
    tmp[0::2] = v0
    tmp[1::2] = v1
    p.add_lines(tmp, color=color)

In [3]:
v, f = igl.read_triangle_mesh("data/bunny.off")
to_pyvista_mesh(v, f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e6bdd65f0_0&reconnect=auto" class="pyvis…

# Vertex normal

In [4]:
#Standard face normal --> I assume this is the standard vertex normals as described in README
vertexNormal = igl.per_vertex_normals(v, f, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_UNIFORM)
plot_mesh_with_normals(v, f, vertexNormal, factor=0.01)


Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e0a2ab370_1&reconnect=auto" class="pyvis…

In [5]:
#Area-weighted face normal
faceareaVertexNormal = igl.per_vertex_normals(v, f, igl.PER_VERTEX_NORMALS_WEIGHTING_TYPE_AREA)
plot_mesh_with_normals(v, f, faceareaVertexNormal, factor=0.01)


Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e6bdd7490_2&reconnect=auto" class="pyvis…

In [ ]:
#Mean-curvature normal
#note to self: reminder to use the cotangent-weighted Laplacian as described in readme
eps = 1e-8 #not told what to use, so pick a threshold value!

cotanLaplace = igl.cotmatrix(v,f)           #the cotangent Laplacian

verticesWithCotanLaplace = cotanLaplace @ v #apply to verteces 

meanCurveMag = np.linalg.norm(verticesWithCotanLaplace, axis=1)#magnitude of mean-curvature normal vectors

meanNormals = np.copy(faceareaVertexNormal)       #for sign comparison

reliableNormalsOnly = meanCurveMag > eps
meanNormals[reliableNormalsOnly] = verticesWithCotanLaplace[reliableNormalsOnly] / meanCurveMag[reliableNormalsOnly][:, None]

#we can get negative values, so we compare to standard normals to get correct direction
signs = np.sign(np.sum(meanNormals * vertexNormal, axis=1))
signs[signs == 0] = 1
meanNormals = meanNormals * signs[:, None]
plot_mesh_with_normals(v, f, meanNormals, factor=0.01)


Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e6c0c7ee0_3&reconnect=auto" class="pyvis…

In [17]:
#PCA normal
numVertices = v.shape[0]
adjacency = [[] for _ in range(numVertices)]
for triangle in f:
    a, b, c = triangle
    adjacency[a].extend([b, c])
    adjacency[b].extend([a, c])
    adjacency[c].extend([a, b])

adjacency = [list(set(n)) for n in adjacency] #ensure no duplicates

#could do this next part with np.argpartition as suggested in assignment description -> bfs chosen for practice
def bfsOnNeighbours(startVertex, adjacency, k): #k is neighbour
    visited = set([startVertex])
    queue = deque([startVertex])
    neighbours = []
    while queue and len(neighbours) < k:
        current = queue.popleft()
        for next in adjacency[current]:
            if next not in visited:
                visited.add(next)
                neighbours.append(next)
                queue.append(next)
                if len(neighbours) >= k:
                    break
    return neighbours

k = 20
pcaNormals = np.zeros_like(v)
for i in range(numVertices):
    nIndex = bfsOnNeighbours(i, adjacency, k)
    theseOnesIndex = [i] + nIndex
    theseOnesPoints = v[theseOnesIndex]
    centre = np.mean(theseOnesPoints, axis=0)
    centredPoints = theseOnesPoints - centre
    #covariance matrix
    covariance = centredPoints.T @ centredPoints
    eigenValues, eigenVectors = np.linalg.eig(covariance)
    #the smallest eigenvalue is the calculated normal direction *** not necessarily correct direction at this point
    smallestIndex = np.argmin(eigenValues)
    normal = eigenVectors[:, smallestIndex]
    #all normals should align with standard vertex normals
    if np.dot(normal, vertexNormal[i]) < 0:
        normal = -normal
    pcaNormals[i] = normal
plot_mesh_with_normals(v, f, pcaNormals, factor=0.01)    


Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e8631d210_4&reconnect=auto" class="pyvis…

In [24]:
#Quadratic fitting normal
quadraticNormals = np.zeros_like(v)
k = 20

for i in range(numVertices):
    nIndex = bfsOnNeighbours(i, adjacency, k)
    theseOnesIndex = [i] + nIndex
    theseOnesPoints = v[theseOnesIndex]
    center = np.mean(theseOnesPoints, axis=0)
    centredPoints = theseOnesPoints - center
    #covariance matrix
    covariance = centredPoints.T @ centredPoints
    eigenValues, eigenVectors = np.linalg.eig(covariance)
    #the smallest eigenvalue is the calculated normal direction *** not necessarily correct direction at this point
    smallestIdx = np.argmin(eigenValues)
    localNormal = eigenVectors[:, smallestIdx]
    #the other two are tangent directions
    tangentIdx = [idx for idx in range(3) if idx != smallestIdx]
    tangent1 = eigenVectors[:, tangentIdx[0]]
    tangent2 = eigenVectors[:, tangentIdx[1]]

    #all normals should align with standard vertex normals
    if np.dot(localNormal, vertexNormal[i]) < 0:
        localNormal = -localNormal
    relativePoints = theseOnesPoints - v[i]

    u = relativePoints @ tangent1
    vv = relativePoints @ tangent2
    h = relativePoints @ localNormal

    # = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =
    #quadratic fit: h(u,v) = a*u^2 + b*u*v + c*v^2 + d*u + e*v + f
    A = np.column_stack([u**2, u * vv, vv**2, u, vv, np.ones_like(u)])
    # = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = = =

    coeffs, _, _, _ = np.linalg.lstsq(A, h, rcond=None)
    a, b, c, d, e, f0 = coeffs

    # local normal at (0,0): (-dh/du, -dh/dv, 1) = (-d, -e, 1)
    localQuadNormal = np.array([-d, -e, 1.0])
    localQuadNormal = localQuadNormal / np.linalg.norm(localQuadNormal)

    #local coordinates back to world coordinates
    worldNormal = ( localQuadNormal[0] * tangent1 + localQuadNormal[1] * tangent2 + localQuadNormal[2] * localNormal )
    worldNormal = worldNormal / np.linalg.norm(worldNormal)

    #all normals should align with standard vertex normals
    if np.dot(worldNormal, vertexNormal[i]) < 0:
        worldNormal = -worldNormal

    quadraticNormals[i] = worldNormal

plot_mesh_with_normals(v, f, quadraticNormals, factor=0.01)

Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e6bdd6fb0_5&reconnect=auto" class="pyvis…

# Curvature

In [30]:
#gaussian curvature

gaussianCurvature = igl.gaussian_curvature(v, f)

massMatrix = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_VORONOI) # Voronoi / barycentric-style vertex area matrix

vertexAreas = massMatrix.diagonal()
safeAreas = np.maximum(vertexAreas, 1e-12)
gaussianCurvature = gaussianCurvature / safeAreas

#plot mesh colored by Gaussian curvature
p = pv.Plotter()
p.add_mesh(to_pyvista_mesh(v, f), show_edges=True, scalars=gaussianCurvature)
p.show()

Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e8d97a020_6&reconnect=auto" class="pyvis…

In [ ]:
# principal curvature

pc = igl.principal_curvature(v, f)

#for debugging = = = = = = = = = = = = 
print("number of outputs:", len(pc))
for i, x in enumerate(pc):
    print(i, type(x), np.shape(x))
#for debugging = = = = = = = = = = = = 

pd1, pd2, k1, k2, _ = igl.principal_curvature(v, f)

k_min = np.minimum(k1, k2)
k_max = np.maximum(k1, k2)

minIsK1 = k1 <= k2 #match directions to min/max values
dir_min = np.where(minIsK1[:, None], pd1, pd2)
dir_max = np.where(minIsK1[:, None], pd2, pd1)

lineLength = 0.01 #scale the line
start_k_min = v
end_k_min = v + lineLength * dir_min

start_k_max = v
end_k_max = v + lineLength * dir_max

#plot mesh 
p = pv.Plotter()
p.add_mesh(to_pyvista_mesh(v, f), show_edges=True, scalars=k_max)
plot_lines(p, start_k_min, end_k_min, "red")
plot_lines(p, start_k_max, end_k_max, "green")
p.show()

number of outputs: 5
0 <class 'numpy.ndarray'> (3485, 3)
1 <class 'numpy.ndarray'> (3485, 3)
2 <class 'numpy.ndarray'> (3485,)
3 <class 'numpy.ndarray'> (3485,)
4 <class 'list'> (0,)


Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27e9d8ac8b0_8&reconnect=auto" class="pyvis…

# Smoothing with the Laplacian

In [43]:
from scipy.sparse.linalg import spsolve
import scipy.sparse as sp

In [ ]:
# Explicit laplacian
vExplicit = v.copy()

#tweak lambdaImplicit & numIterations 
lambdaExplicit = 0.000001
numIterations = 1 

C = igl.cotmatrix(v, f) #cotangent matrix
M = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_VORONOI) #mass matrix
MDiag = np.maximum(M.diagonal(), 1e-12)
Minv = sp.diags(1.0 / M.diagonal())

L = Minv @ C #laplacian operator

for _ in range(numIterations):
    vExplicit = vExplicit - lambdaExplicit * (L @ vExplicit)

to_pyvista_mesh(vExplicit, f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27efbac9660_32&reconnect=auto" class="pyvi…

In [ ]:
# Implicit laplacian
vImplicit = v.copy()

#tweak lambdaImplicit & numIterations
lambdaImplicit = 0.000001
numIterations = 10

C = igl.cotmatrix(v, f)
M = igl.massmatrix(v, f, igl.MASSMATRIX_TYPE_VORONOI)

A = M - lambdaImplicit * C

for _ in range(numIterations):
    b = M @ vImplicit

    x = spsolve(A, b[:, 0])
    y = spsolve(A, b[:, 1])
    z = spsolve(A, b[:, 2])

    vImplicit = np.column_stack((x, y, z))

to_pyvista_mesh(vImplicit, f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:56369/index.html?ui=P_0x27f2c163e80_33&reconnect=auto" class="pyvi…